In [1]:
import csv
import re
from datetime import datetime

# CONFIGURAÇÃO DOS CAMINHOS DOS ARQUIVOS
produtos = 'olist_products_dataset.csv'
pedidos = 'olist_orders_dataset.csv'

# LIMPEZA E PADRONIZAÇÃO
def limpar_categoria(texto):
    if not texto or texto.strip() == "":
        return "sem categoria"

    # .strip() remove espaços nas pontas, .lower() padroniza para minúsculo
    texto_limpo = texto.strip().lower()

    # Regex: substitui espaços ou hífens por underline
    texto_limpo = re.sub(r'[\s-]+', '_', texto_limpo)

    # Regex: remove qualquer caractere que não seja letra, número ou underline
    texto_limpo = re.sub(r'[^\w]', '', texto_limpo)

    return texto_limpo

# FORMATAÇÃO DATETIME
def formatar_data(string_data):
    if not string_data or string_data.strip() == "":
        return "Data Indisponível"
    try:
        # Lê o formato original
        data = datetime.strptime(string_data.strip(), "%Y-%m-%d %H:%M:%S")
        # Converte para o formato br
        return data.strftime("%d/%m/%Y")
    except ValueError:
        return "Data Indisponível"

# PIPELINE DE PRODUTOS
produtos_limpos = []
total_nulos_corrigidos_prod = 0

with open(produtos, mode='r', encoding='utf-8') as prod:
    leitor_produto = csv.DictReader(prod)

    for linha in leitor_produto:
        # Tratamento da Categoria
        cat_original = linha.get('product_category_name', '')
        
        # Correção do contador: Verifica estritamente se estava nulo/vazio
        if not cat_original or cat_original.strip() == "":
            linha['product_category_name'] = "sem categoria"
            total_nulos_corrigidos_prod += 1
        else:
            linha['product_category_name'] = limpar_categoria(cat_original)

        # Tratamento das dimensões físicas (peso, comprimento, etc.)
        # Justificativa técnica: Preenchemos com '0' para manter a base íntegra.
        dimensoes = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
        for dim in dimensoes:
            valor_dim = linha.get(dim, '')
            if not valor_dim or valor_dim.strip() == "":
                linha[dim] = '0'
                total_nulos_corrigidos_prod += 1

        produtos_limpos.append(linha)

# PIPELINE DE PEDIDOS
pedidos_limpos = []
total_pedidos = 0
pedidos_cancelados = 0
entrega_nula_nao_cancelado = 0  # Variável para testar a hipótese de negócio

with open(pedidos, mode='r', encoding='utf-8') as ped:
    leitor_pedidos = csv.DictReader(ped)

    for linha in leitor_pedidos:
        total_pedidos += 1
        status = linha.get('order_status', '')
        data_entrega = linha.get('order_delivered_customer_date', '').strip()

        # Contagem de cancelados
        if status == 'canceled':
            pedidos_cancelados += 1

        # Lógica para comprovar a hipótese de negócio
        if not data_entrega:
            if status != 'canceled':
                # Conta pedidos sem entrega que NÃO foram cancelados
                entrega_nula_nao_cancelado += 1

        # Formatação de data
        linha['order_approved_at'] = formatar_data(linha.get('order_approved_at', ''))

        pedidos_limpos.append(linha)

# EXIBIÇÃO DOS RESULTADOS (SUMÁRIO ESTATÍSTICO MANUAL)
print("="*60)
print("                    RELATÓRIO/STATUS")
print("="*60)
print(f"Total de linhas de produtos processadas: {len(produtos_limpos)}")
print(f"Total de registros nulos corrigidos (Produtos): {total_nulos_corrigidos_prod}")
print("-" * 60)
print(f"Total de linhas de pedidos processadas:  {total_pedidos}")
print(f"Total de pedidos cancelados identificados: {pedidos_cancelados}")
print("-" * 60)
print("VALIDAÇÃO DA HIPÓTESE DE NEGÓCIO:")
if entrega_nula_nao_cancelado == 0:
    print("Hipótese CONFIRMADA: As datas de entrega estão nulas obrigatoriamente devido ao cancelamento.")
else:
    print(f"Hipótese REJEITADA: Há {entrega_nula_nao_cancelado} pedidos sem data de entrega que NÃO estão cancelados.")
    print("Motivo: Pedidos ainda podem estar em rota, faturados ou em processamento.")
print("="*60)

                    RELATÓRIO/STATUS
Total de linhas de produtos processadas: 32951
Total de registros nulos corrigidos (Produtos): 618
------------------------------------------------------------
Total de linhas de pedidos processadas:  99441
Total de pedidos cancelados identificados: 625
------------------------------------------------------------
VALIDAÇÃO DA HIPÓTESE DE NEGÓCIO:
Hipótese REJEITADA: Há 2346 pedidos sem data de entrega que NÃO estão cancelados.
Motivo: Pedidos ainda podem estar em rota, faturados ou em processamento.
